# Anticipez les besoins en consommation de bâtiments — Modélisation

Suite de `01_analyse_exploratoire.ipynb` : on repart du jeu nettoyé (1 454 bâtiments non résidentiels, relevés conformes et cohérents,
12 colonnes utiles plus la cible).

Ce notebook crée d'abord de **nouvelles colonnes** (*features*) à partir des colonnes existantes, pour donner au futur modèle des informations
plus parlantes (âge du bâtiment, surface par étage, présence d'un restaurant…). Il prépare ensuite ces données, compare plusieurs modèles,
puis optimise et interprète le meilleur.

#### Petit lexique

| Terme | Signification |
|---|---|
| **Feature** | Une information donnée au modèle pour prédire (une colonne) |
| **Cible** | Ce que le modèle doit prédire : ici la consommation totale `SiteEnergyUse(kBtu)` |
| **Feature engineering** | Fabriquer de nouvelles features à partir des colonnes existantes |
| **Feature binaire** | Feature qui vaut 1 (oui) ou 0 (non), par exemple « le bâtiment contient un restaurant » |
| **Modalité** | Une valeur possible d'une colonne texte (ex. `Hotel` est une modalité de `PrimaryPropertyType`) |
| **Cardinalité** | Nombre de modalités d'une colonne texte |
| **Fuite de données** | Utiliser pour prédire une information qui ne sera pas connue au moment de la prédiction. Le modèle paraît excellent mais est inutilisable en pratique |
| **Log** | Logarithme : transforme 10, 100, 1 000 en 1, 2, 3. Resserre les valeurs très étalées |
| **Imputer** | Remplacer une valeur manquante par une valeur estimée (médiane, valeur la plus fréquente…) |
| **pied²** | Unité de surface américaine (1 pied² ≈ 0,093 m²) |

# Modélisation

### Import des modules

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

### Chargement du jeu nettoyé

In [ ]:
df = pd.read_csv("../data/building_consumption_clean.csv")
cible = "SiteEnergyUse(kBtu)"
print(df.shape)
df.head()

On retrouve bien les **1 454 bâtiments** et les **13 colonnes** exportés par le notebook 01 : 12 informations sur le bâtiment
(usage, quartier, année, étages, surfaces, présence de gaz et de vapeur) et la cible `SiteEnergyUse(kBtu)`.

### Feature Engineering

**Règle suivie pour chaque feature** : elle doit pouvoir être calculée pour un bâtiment **jamais mesuré**.
On n'utilise donc jamais une valeur issue du relevé de consommation (le notebook 01 les a déjà retirées, à l'exception de la cible).

Les features couvrent cinq catégories d'informations : **temporalité**, **structure**, **usages**, **sources d'énergie** et **localisation**.
Toutes sont calculées ligne par ligne, à partir du bâtiment lui-même, ou avec des règles fixées à l'avance (seuils, regroupements) :
aucune n'utilise une statistique calculée sur l'ensemble du jeu, ce qui évite une fuite entre futurs jeux d'entraînement et de test.

#### 1. Temporalité

- `BuildingAge` : âge du bâtiment au moment du relevé (2016). Plus parlant que l'année et directement interprétable.
- `PeriodeConstruction` : tranches correspondant à de grandes époques constructives — avant 1946 (bâti ancien), 1946-1979 (après-guerre,
  avant les premières réglementations énergétiques consécutives aux chocs pétroliers), 1980-2000, après 2000 (normes plus récentes).
  Les tranches permettent à un modèle linéaire de capter un effet **non linéaire** de l'âge.

In [ ]:
ANNEE_RELEVE = 2016
df["BuildingAge"] = ANNEE_RELEVE - df["YearBuilt"]
df["PeriodeConstruction"] = pd.cut(
    df["YearBuilt"],
    bins=[1899, 1945, 1979, 2000, ANNEE_RELEVE],
    labels=["Avant 1946", "1946-1979", "1980-2000", "Après 2000"],
)
df["PeriodeConstruction"].value_counts().sort_index()

Nombre de bâtiments par période de construction : 433 avant 1946, 520 entre 1946 et 1979, 311 entre 1980 et 2000, 190 après 2000.
Chaque période contient assez de bâtiments pour qu'un modèle puisse apprendre son effet. Les deux tiers du parc datent d'avant 1980,
c'est-à-dire d'avant les premières normes énergétiques.

#### 2. Structure du bâtiment

- `LogSurfaceTotale` : log de la surface totale. L'analyse exploratoire a montré une relation **linéaire en log-log** entre surface et consommation.
- `PartParking` : part de la surface totale occupée par du parking, peu consommateur (éclairage, ventilation) comparé aux bureaux ou commerces.
- `SurfaceParEtage` : surface bâtie moyenne par étage — distingue un entrepôt de plain-pied très étendu d'une tour compacte.
- `SurfaceParBatiment` : surface bâtie moyenne par bâtiment de la propriété.
- `MultiBatiments` : 1 si la propriété compte plusieurs bâtiments (campus, hôpitaux…).

Pas de risque de division par zéro : l'analyse exploratoire a supprimé les bâtiments à 0 étage et corrigé les propriétés à 0 bâtiment.

In [ ]:
df["LogSurfaceTotale"] = np.log(df["PropertyGFATotal"])
df["PartParking"] = df["PropertyGFAParking"] / df["PropertyGFATotal"]
df["SurfaceParEtage"] = df["PropertyGFABuilding(s)"] / df["NumberofFloors"]
df["SurfaceParBatiment"] = df["PropertyGFABuilding(s)"] / df["NumberofBuildings"]
df["MultiBatiments"] = (df["NumberofBuildings"] > 1).astype(int)

df[["LogSurfaceTotale", "PartParking", "SurfaceParEtage", "SurfaceParBatiment", "MultiBatiments"]].describe().T.round(2)

Le profil des 5 nouvelles features de structure (moyenne, dispersion, minimum, quartiles, maximum) :
- `LogSurfaceTotale` varie de 9,3 à 14,6 : le log a **resserré** des surfaces qui allaient de 11 000 à 2,2 millions de pieds².
- `PartParking` : médiane 0, la plupart des bâtiments n'ont pas de parking ; en moyenne, le parking occupe 7 % de la surface.
- `SurfaceParEtage` : médiane d'environ 20 000 pieds² (≈ 1 900 m²) par étage, mais jusqu'à 1,1 million pour un très grand bâtiment de plain-pied.
- `MultiBatiments` : moyenne 0,03, donc seulement **3 %** des propriétés comptent plusieurs bâtiments.

#### 3. Usages du bâtiment

**Nombre d'usages.** `ListOfAllPropertyUseTypes` liste les usages séparés par des virgules.
Piège : certains libellés contiennent eux-mêmes des virgules entre parenthèses, par exemple
`Repair Services (Vehicle, Shoe, Locksmith, etc)`. Un simple `split(",")` compterait 4 usages au lieu d'un :
on retire d'abord le contenu des parenthèses.

In [ ]:
def lister_usages(texte):
    """Renvoie la liste des usages d'un bâtiment à partir de ListOfAllPropertyUseTypes."""
    if pd.isna(texte):
        return []
    texte = re.sub(r"\([^)]*\)", "", texte)  # retire « (Vehicle, Shoe, …) » qui contient des virgules
    return [usage.strip() for usage in texte.split(",") if usage.strip()]

usages = df["ListOfAllPropertyUseTypes"].apply(lister_usages)

# Vérification du piège : sans retirer les parenthèses, on compterait trop d'usages
naif = df["ListOfAllPropertyUseTypes"].fillna("").str.split(",").str.len()
print("Bâtiments dont le comptage naïf serait faux :", (naif != usages.str.len()).sum())
print("Liste des usages absente :", df["ListOfAllPropertyUseTypes"].isna().sum())

**19 bâtiments** auraient eu un nombre d'usages faux avec un simple découpage aux virgules : le piège est réel, la précaution est utile.
Aucun bâtiment n'a de liste d'usages absente.

Par sécurité (bâtiment futur sans liste), on borne `NbUsages` à 1 : un bâtiment a au moins un usage, son `PrimaryPropertyType`.

In [ ]:
df["NbUsages"] = usages.str.len().clip(lower=1)
df["MultiUsage"] = (df["NbUsages"] > 1).astype(int)
df["NbUsages"].value_counts().sort_index()

Nombre de bâtiments selon leur nombre d'usages : 651 n'ont qu'un seul usage, 469 en ont deux, 202 en ont trois, et quelques-uns jusqu'à 11.
Plus de la moitié des bâtiments sont donc **mixtes** (`MultiUsage` = 1).

**Part de l'usage principal.** Indique si le bâtiment est mono-usage (part proche de 1) ou mixte.
La part est bornée à 1 : l'analyse exploratoire a montré que la surface d'un usage peut légèrement dépasser la surface totale (écart ≤ 10 % conservé).
Elle reste manquante pour les quelques bâtiments sans usage principal renseigné : elle sera imputée lors de la préparation.

In [ ]:
df["PartUsagePrincipal"] = (df["LargestPropertyUseTypeGFA"] / df["PropertyGFATotal"]).clip(upper=1)
df["PartUsagePrincipal"].describe().round(2)

Médiane de 0,92 : dans la moitié des bâtiments, l'usage principal occupe plus de 92 % de la surface. Un quart des bâtiments
est en revanche nettement mixte (part inférieure à 0,66). Le `count` est de 1 450 sur 1 454 : 4 bâtiments n'ont pas d'usage principal renseigné.

**Présence d'usages très énergivores.** Un bâtiment de bureaux qui abrite un restaurant, un data center ou une piscine consommera plus
qu'un bureau classique. On crée une feature binaire par grande famille d'usage intensif, à partir de **tous** les usages du bâtiment
(pas seulement le principal).

In [ ]:
usages_intensifs = {
    "UsageRestauration": ["Restaurant", "Fast Food Restaurant", "Food Service", "Other - Restaurant/Bar", "Bar/Nightclub"],
    "UsageAlimentaire": ["Supermarket/Grocery Store", "Food Sales", "Convenience Store without Gas Station", "Wholesale Club/Supercenter"],
    "UsageSante": ["Hospital", "Other/Specialty Hospital", "Urgent Care/Clinic/Other Outpatient", "Laboratory"],
    "UsageDataCenter": ["Data Center"],
    "UsagePiscineSport": ["Swimming Pool", "Fitness Center/Health Club/Gym"],
    "UsageFroid": ["Refrigerated Warehouse"],
    "UsageParking": ["Parking"],
}
for colonne, liste in usages_intensifs.items():
    df[colonne] = usages.apply(lambda u: int(any(usage in liste for usage in u)))

df[list(usages_intensifs)].sum().to_frame("Nb bâtiments")

Nombre de bâtiments qui contiennent chaque usage énergivore : 393 ont un **parking**, 127 un espace de **restauration**,
61 un commerce **alimentaire**, 43 un **data center**, 39 un usage de **santé**, 39 une **piscine ou salle de sport**
et 16 un **entrepôt frigorifique**.

**Compression d'une catégorie à haute cardinalité.** `PrimaryPropertyType` compte une vingtaine de modalités, dont certaines
avec une dizaine de bâtiments seulement (hôpitaux, laboratoires) : trop peu pour qu'un modèle apprenne un effet fiable.
On les regroupe en **familles d'usage** proches en termes de consommation (cf. intensités vues en analyse exploratoire).
Le regroupement est une **règle métier fixée à l'avance**, pas un seuil de fréquence calculé sur les données.

In [ ]:
familles = {
    "Bureau": ["Large Office", "Small- and Mid-Sized Office", "Office"],
    "Santé": ["Hospital", "Medical Office", "Laboratory", "Senior Care Community"],
    "Alimentation": ["Supermarket / Grocery Store", "Restaurant"],
    "Commerce": ["Retail Store"],
    "Logistique": ["Warehouse", "Distribution Center", "Refrigerated Warehouse", "Self-Storage Facility"],
    "Enseignement": ["K-12 School", "University"],
    "Hébergement": ["Hotel", "Residence Hall"],
    "Culte": ["Worship Facility"],
    "Mixte / Autre": ["Mixed Use Property", "Other"],
}
type_vers_famille = {t: famille for famille, types in familles.items() for t in types}
df["FamilleUsage"] = df["PrimaryPropertyType"].map(type_vers_famille)

assert df["FamilleUsage"].notna().all(), "Un type de bâtiment n'a pas de famille"
df["FamilleUsage"].value_counts()

Nombre de bâtiments par famille d'usage. Chaque famille contient désormais au moins 51 bâtiments, alors que certains types d'origine
n'en avaient que 10 (hôpitaux, laboratoires). Les bureaux (443) sont la famille la plus représentée, devant les usages mixtes (335)
et la logistique (267). La ligne `assert` vérifie que chaque type de bâtiment a bien été rangé dans une famille : si un type était oublié,
le notebook s'arrêterait avec une erreur.

#### 4. Sources d'énergie

Les **quantités** de gaz, d'électricité ou de vapeur ne peuvent pas être utilisées : elles composent directement la cible (fuite de données).
Le notebook 01 les a donc remplacées par deux indicateurs de **présence** : `UtiliseGaz` (bâtiment équipé au gaz) et `UtiliseVapeur`
(bâtiment raccordé au réseau de vapeur urbain). Ce sont des informations **structurelles** : elles dépendent des équipements du bâtiment,
connus sans relevé de consommation, et pas de l'intensité de la consommation.

À partir de ces deux indicateurs, on crée `NbSourcesEnergie` : le nombre de sources d'énergie du bâtiment (l'électricité, présente
partout, plus le gaz et la vapeur). Un bâtiment qui combine plusieurs sources a souvent des besoins de chauffage ou de process plus importants.

In [ ]:
df["NbSourcesEnergie"] = 1 + df["UtiliseGaz"] + df["UtiliseVapeur"]
print(df[["UtiliseGaz", "UtiliseVapeur"]].mean().round(2).to_frame("Part des bâtiments"), "\n")
df["NbSourcesEnergie"].value_counts().sort_index()

- **71 %** des bâtiments sont équipés au gaz et **7 %** sont raccordés au réseau de vapeur.
- `NbSourcesEnergie` : 371 bâtiments n'utilisent que l'électricité, 1 024 combinent deux sources et 59 en combinent trois.

#### 5. Localisation

`Neighborhood` compte 13 quartiers, dont certains avec une trentaine de bâtiments seulement. On les regroupe en trois grandes zones
géographiques de Seattle, fixées à l'avance d'après la carte de la ville :
- **Centre** : Downtown, Lake Union, East, Central — le cœur de la ville, où se concentrent les tours de bureaux, les hôtels et les hôpitaux ;
- **Nord** : North, Northeast, Northwest, Ballard, Magnolia / Queen Anne — quartiers plutôt résidentiels ;
- **Sud** : Greater Duwamish, Southeast, Southwest, Delridge — avec la grande zone industrielle et portuaire.

In [ ]:
zones = {
    "Centre": ["DOWNTOWN", "LAKE UNION", "EAST", "CENTRAL"],
    "Nord": ["NORTH", "NORTHEAST", "NORTHWEST", "BALLARD", "MAGNOLIA / QUEEN ANNE"],
    "Sud": ["GREATER DUWAMISH", "SOUTHEAST", "SOUTHWEST", "DELRIDGE"],
}
quartier_vers_zone = {q: zone for zone, quartiers in zones.items() for q in quartiers}
df["ZoneVille"] = df["Neighborhood"].map(quartier_vers_zone)

assert df["ZoneVille"].notna().all(), "Un quartier n'a pas de zone"
pd.crosstab(df["FamilleUsage"], df["ZoneVille"])

Tableau croisé famille d'usage × zone : chaque case donne le nombre de bâtiments.
- Le **Centre** concentre les **bureaux** (284 sur 443) et l'**hébergement** (57 sur 86).
- Le **Sud** concentre la **logistique** (193 sur 267), autour de la zone industrielle de Greater Duwamish.
- Le **Nord** regroupe davantage les lieux de **culte** (40 sur 63).

On retrouve ce que l'analyse exploratoire avait montré : la zone et l'usage sont liés. La ligne `assert` vérifie que chaque quartier a bien une zone.

#### Récapitulatif et contrôles

In [ ]:
nouvelles_features = [
    "BuildingAge", "PeriodeConstruction",
    "LogSurfaceTotale", "PartParking", "SurfaceParEtage", "SurfaceParBatiment", "MultiBatiments",
    "NbUsages", "MultiUsage", "PartUsagePrincipal", *usages_intensifs, "FamilleUsage",
    "NbSourcesEnergie",
    "ZoneVille",
]
numeriques = df[nouvelles_features].select_dtypes("number")
controle = pd.DataFrame({
    "type": df[nouvelles_features].dtypes.astype(str),
    "manquants": df[nouvelles_features].isna().sum(),
    "infinis": np.isinf(numeriques).sum().reindex(nouvelles_features, fill_value=0),
    "nb valeurs": df[nouvelles_features].nunique(),
})
controle

Contrôle des 20 nouvelles features : leur type, le nombre de valeurs manquantes, le nombre de valeurs infinies (qui apparaîtraient en cas
de division par zéro) et le nombre de valeurs différentes.
- Aucune valeur infinie : pas de division par zéro.
- Seule `PartUsagePrincipal` a 4 valeurs manquantes (usage principal non renseigné), à imputer lors de la préparation.
- Les features binaires ont bien 2 valeurs (0 et 1), `FamilleUsage` en a 9 et `ZoneVille` 3.

In [ ]:
features_correlation = numeriques.join(df[["UtiliseGaz", "UtiliseVapeur"]])
corr_cible = (
    features_correlation.corrwith(np.log(df[cible]), method="spearman")
    .sort_values()
)
fig, ax = plt.subplots(figsize=(8, 7))
corr_cible.plot(kind="barh", ax=ax)
ax.set_title("Corrélation de Spearman des features avec log(consommation)")
plt.show()
corr_cible.round(2)

Une barre par feature numérique : sa corrélation avec le log de la consommation, de -1 à 1. Une barre longue vers la droite
indique que la consommation augmente avec la feature ; une barre vers la gauche qu'elle diminue ; une barre courte, qu'il y a peu de lien.

Ce premier aperçu ne remplace pas l'entraînement d'un modèle :
- `LogSurfaceTotale` (0,74) reste de loin la plus liée à la consommation ; `SurfaceParBatiment` (0,69) lui est très proche → redondance probable.
- `BuildingAge` est **négativement** liée (-0,29) : les bâtiments récents sont en moyenne plus grands.
- Les indicateurs d'usages et de sources d'énergie ont des corrélations faibles (0,1 à 0,3) **prises isolément** : leur effet joue
  **à surface égale** (un restaurant consomme plus qu'un entrepôt de même taille), ce qu'une corrélation seule ne montre pas.

La sélection finale (suppression des features redondantes) est faite dans la partie suivante.

| Catégorie | Features créées | Colonnes sources | Fuite ? |
|---|---|---|---|
| Temporalité | `BuildingAge`, `PeriodeConstruction` | `YearBuilt` | Non |
| Structure | `LogSurfaceTotale`, `PartParking`, `SurfaceParEtage`, `SurfaceParBatiment`, `MultiBatiments` | surfaces, étages, nombre de bâtiments | Non |
| Usages | `NbUsages`, `MultiUsage`, `PartUsagePrincipal`, `Usage*` (7 binaires), `FamilleUsage` | liste et surface des usages, type principal | Non |
| Sources d'énergie | `UtiliseGaz`, `UtiliseVapeur` (notebook 01), `NbSourcesEnergie` | présence de gaz / vapeur, **jamais la quantité** | Non |
| Localisation | `ZoneVille` | `Neighborhood` | Non |

### Préparation des features pour la modélisation

In [ ]:
# CODE PREPARATION DES FEATURES

### Comparaison de différents modèles supervisés

In [ ]:
# CODE COMPARAISON DES MODELES

### Optimisation et interprétation du modèle

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE